# P-2026-11-28-1: Do the de-duplicated denominator (DD) and the ch. 89 fix (F1) work together?

**Question** (from `proposals/2026-11-27.md` #1, in the open 11-27 PR): two grouped-ECI fixes have passed the pre-stated rule A so far. **F1** moves HS ch. 89 (ships, floating platforms) from *Daily Instruments* to *Transportation* (11-24, passed on the rebuilt base, 3 of 3 check years). **DD** computes the RCA denominators over the 17 non-aggregate columns, so the *High-complexity Products* aggregate stops double-counting its chapters (11-27). DD's pass was weak: it held on the cleaned base and failed on the rebuilt base by a 2019 ΔM2 of −0.006. Tonight's three questions: does **DF = DD + F1** pass rule A against the lab baseline? Does it beat **each fix alone**? Is the combination additive, or do the two fixes overlap through the same threshold flips?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`. Open nightly PRs for 11-20 to 11-27 each took one proposal, and no run folder on `main` or any `claude/nightly-*` branch names 11-27 #1. It is the most decision-relevant proposal on the list. If Will adopts grouped fixes for Fall 2026, he needs to know whether the only two candidates that passed can be adopted *together*. It needs no `lib/` change and no new data.

- **Date:** 2026-11-28 (America/New_York), nightly run. ID `P-2026-11-28-1`.
- **Data:** `data/raw/2019–2023_mega.csv` (HS6; every grouped variant on the primary base is rebuilt from this with `lib/research.groups_from_raw`; also the HS4 base that defines the sample). `data/cleaned/2019–2023clean.csv` (secondary base C*, via `lib/research.lab_groups_iso3`, EU-28 dropped). `external-data/raw/atlas_hs92_country_year.csv` (Growth Lab published ECI, benchmark only). `external-data/raw/wdi_panel.csv` (GDPpc PPP). No downloads.
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1) and excluded from every rule.
- **Library:** `lib/eci.py` and `lib/research.py`, imported unchanged. `o4`, `lab_M`, the joint common sample, F1 and DD are copied from 11-24 to 11-27 unchanged, so the numbers can be compared directly.
- **Run time:** printed in the last cell.

## TL;DR

- **DD + F1 (DF) fails rule A on the primary base, and adding DD to F1 makes F1 worse.** On the rebuilt base R, DF against the lab baseline gives 2019 ΔM1 **+0.028** [−0.038, 0.101] but ΔM2 **−0.010** [−0.109, 0.080]. It is up on both metrics in 3 of 3 check years, so it fails on 2019 alone. F1 alone gives **+0.037 / +0.034** in 2019, which reproduces 11-25 exactly. So stacking DD on F1 costs **−0.009 (M1) and −0.044 (M2)** in 2019. On the cleaned base C\*, DF passes against R0 (+0.047 / +0.022), but **DF never passes against F1 alone on either base** (2019 ΔM2 −0.044 on R, −0.012 on C\*).
- **The two fixes are sub-additive: they partly fix the same cells.** Interaction = ΔDF − (ΔDD + ΔF1) is negative in every 2019–2021 base-year on both metrics. The 2019–2022 mean is **−0.016 (M1) and −0.022 (M2)** on R, and −0.016 / −0.019 on C\*. DF beats the better single fix on both metrics in only **1 of 4** years on each base.
- **The overlap comes from threshold flips, not the denominators.** F1 leaves DD's per-country factor exactly unchanged (max difference 0, checked in every base-year). Of the 83 + 14 cells DD and F1 flip in 2019 (R), **10 cancel** in DF and only **1** new cell appears. All 10 are in the two columns F1 touches (7 *Daily Instruments*, 3 *Transportation*), Vietnam's *Daily Instruments* cell among them. Vietnam's 2019 rank is 31 (R0), 54 (DD), 32 (F1) and **38 (DF)**. ATLAS puts it at 56.
- **For the reports:** DF changes **7 of 28** peer positions for the Spring 2026 countries in 2019–2022, against 2 of 28 for F1 and 4 of 28 for DD. DF and F1 each match the ATLAS peer position in **19 of 28** cases (R0 and DD: 17). Rank stability is the same for all four variants (fixed-panel mean 0.946–0.949).
- **Recommendation: F1 alone remains the only fix that passes robustly.** It passes on both bases, with the only 2019 bootstrap CI that excludes 0 (ΔM2 [0.004, 0.076] on R). DD should not be adopted together with F1 on this evidence. Whether to adopt DD on its own is still Will's call, and it rests on the base-dependent pass from 11-27.

## Why this matters for the lab

Every country report prints a grouped-ECI rank and compares it with three peers. Since 11-19, the nightly runs have scored about 20 candidate fixes to the lab's grouped ECI under one pre-stated rule. Two survived: F1, a mapping fix in `datacleaning2025.ipynb`, and DD, a one-line change to `compute_mcp` in `Spring2026.ipynb`. They touch different parts of the pipeline, so the lab could adopt both for Fall 2026. Before recommending that, we need to know three things: whether the combination is still an improvement, whether it beats the stronger single fix (F1), and whether each fix's gain survives the other. This tests our **implementation** (agenda §3, the HS mapping and the circular aggregate) under the pre-registered evaluation of agenda §5. It doesn't test whether ECI predicts growth.

## Method (pre-stated before the first execution)

**Base rules:** the lab's grouped ECI (drop-any-zero filter, strict `RCA > 1`, sorted SVD, O4 orientation), scored on the **joint common sample** of 11-21 to 11-27 (cleaned grouped base ∩ HS4 base ∩ ATLAS ∩ WDI ∩ resource share). Tonight the sample is further restricted to countries that survive the filter under **all four** variants, so every comparison uses the same countries. This is needed for the additivity test.

**Bases.**
- **R (primary)** = the lab mapping rebuilt from `data/raw`. F1 is a chapter remap, so it can only be built exactly from HS chapters. This is the base where F1 passed (11-24/11-25) and where DD *failed* (11-27).
- **C\* (secondary)** = the cleaned CSV the notebook actually reads (DD's passing base). F1 is applied by moving each country's raw ch. 89 exports from *Daily Instruments* to *Transportation*. It is a hybrid of two sources, so any cell that goes negative is clipped to 0, and the number of such cells is reported.

**Variants** (2 × 2):
- **R0** = lab mapping + lab denominators (baseline)
- **DD** = lab mapping + de-duplicated denominators (11-27)
- **F1** = ch. 89 → *Transportation* + lab denominators (11-24)
- **DF** = F1 mapping + DD denominators (**primary variant**)

**Metrics and rule** (unchanged since 11-19): **M1** = Spearman with ATLAS ECI. **M2** = Spearman with log GDPpc (PPP). **Rule A:** a variant beats a baseline only if it is higher on both M1 and M2 in 2019 **and** in ≥ 2 of 2020–2022. 2019 paired country bootstrap, 2,000 draws.

**Pre-stated decisions:**
1. **DF vs R0** (primary): does the combination pass?
2. **DF vs F1**: recommend adopting DD *on top of* F1 only if DF passes rule A with F1 as the baseline.
3. **DF vs DD**: the same check for F1 on top of DD.
4. **Additivity:** interaction = ΔDF − (ΔDD + ΔF1), all vs R0, per metric and year. Cell level: is DF's set of flipped M cells the union of DD's and F1's? Analytic expectation: F1 moves dollars between two *non-aggregate* columns, so the 17-column totals, and with them DD's per-country factor (1 + a_c)/(1 + a_w), should be unchanged. Any interaction must then come from threshold flips and the eigenvector, not from the denominators.

**Report only:** M3 = Spearman with our HS4 ECI. Year-to-year rank stability on a fixed panel. Movers. Spring 2026 report-country peer positions (the same peer dict as Q02 / 11-14 / 11-17 / 11-25 to 11-27).

**Transparency note:** no prototype was run before this notebook. The decisions above were written before the first execution.

## Setup

In [1]:
import sys, time, copy, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2019, 2020, 2021, 2022]
AGG, DI, TR = 'High-complexity Products', 'Daily Instruments', 'Transportation'
PEERS = {'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
         'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'], 'VNM': ['SGP', 'MYS', 'PHL']}

def o4(M):
    """Sorted SVD of binary M, O4 orientation (PCI negative with ubiquity). Copied unchanged from 11-24..11-27."""
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)
    s4 = -np.sign(np.corrcoef(q, kp)[0, 1]); k, q = k * s4, q * s4
    e = pd.Series(k, index=M.index); z = (e - e.mean()) / e.std()
    return z, e, pd.Series(q, index=M.columns), S[1]

def rca_lab(X):
    """D0: lab compute_mcp denominators (totals over all 18 columns, aggregate double-counted)."""
    return eci.rca(X)

def rca_dedup(X):
    """DD (11-27): same 18 columns, country and world totals over the 17 NON-aggregate columns only."""
    X = X.loc[X.sum(1) > 0, X.sum(0) > 0]; base = X.drop(columns=[AGG])
    return X.div(base.sum(1), axis=0) / (X.sum(0) / base.to_numpy().sum())

def lab_M(X, f):
    """Lab rules: drop-any-zero filter on the 18-column matrix, strict RCA > 1."""
    X = eci.notebook_filter(X); Mm = (f(X) > 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def hs_M(X):
    Mm = (eci.rca(X) >= 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def rank(e, idx):
    return e[idx].rank(ascending=False)

# Mappings: L0 = lab mapping; F1 = ch. 89 (ships) moved from Daily Instruments to Transportation (11-24).
L0 = R.LAB_GROUPS
F1 = copy.deepcopy(L0); F1[DI] = [c for c in L0[DI] if c != 89]; F1[TR] = L0[TR] + [89]
print('F1 Daily Instruments:', F1[DI], '| F1 Transportation:', F1[TR], '| ch. 89 in aggregate?', 89 in L0[AGG])

# Benchmarks and joint common sample: identical definition to 11-21..11-27.
G = {y: R.lab_groups_iso3(y, drop_eu=True) for y in YEARS}
XH = {y: R.repo_hs(y, 4) for y in YEARS}
RSH = {y: XH[y][[c for c in XH[y].columns if c[:2] in ['26', '27', '71']]].sum(1) / XH[y].sum(1) for y in YEARS}
AT = R.atlas_cy(); ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
GB = {y: o4(lab_M(G[y], rca_lab)) for y in YEARS}; HB = {y: o4(hs_M(XH[y])) for y in YEARS}
COMMON = {y: sorted(set(GB[y][0].index) & set(HB[y][0].index) & set(ATL[y].dropna().index) & set(LNY[y].index) & set(RSH[y].index)) for y in YEARS}
print('joint common sample n (same definition as 11-21..11-27):', {y: len(COMMON[y]) for y in YEARS})

F1 Daily Instruments: [90, 91, 93, 94, 95] | F1 Transportation: [85, 86, 87, 88, 98, 89] | ch. 89 in aggregate? False


joint common sample n (same definition as 11-21..11-27): {2019: 105, 2020: 105, 2021: 94, 2022: 105, 2023: 109}


In [2]:
# Bases. R (PRIMARY): rebuilt from data/raw with each mapping (F1 needs HS chapters, which the cleaned CSV no longer has).
# C* (SECONDARY): cleaned CSV, with each country's raw ch. 89 exports moved from Daily Instruments to Transportation.
def raw_ch(y, chs):
    """Raw exports of the given HS chapters by ISO3 (same aggregation path as research.groups_from_raw)."""
    return R.groups_from_raw(y, {'x': chs})['x']

XB = {}
for y in YEARS:
    XB['R', 'L0', y] = R.groups_from_raw(y, L0); XB['R', 'F1', y] = R.groups_from_raw(y, F1)
    c = G[y].copy(); c89 = raw_ch(y, [89]).reindex(c.index).fillna(0.0)
    XB['C', 'L0', y] = c
    cf = c.copy(); cf[DI] = c[DI] - c89; cf[TR] = c[TR] + c89; XB['C', 'F1', y] = cf
rows = []
for y in YEARS:
    c, c89 = G[y], raw_ch(y, [89]).reindex(G[y].index).fillna(0.0)
    neg = (c[DI] - c89 < -1.0)
    rows.append({'year': y, 'C* countries': len(c), 'raw ch.89 world $B': c89.sum() / 1e9, 'cleaned DI world $B': c[DI].sum() / 1e9,
                 'C* DI < 0 (n)': int(neg.sum()), 'C* DI < 0 (who)': ', '.join(c.index[neg][:8]),
                 'R: DI==0 under F1 but >0 under L0 (n)': int(((XB['R', 'F1', y][DI] == 0) & (XB['R', 'L0', y][DI] > 0)).sum())})
print(pd.DataFrame(rows).round(1).to_string(index=False))
for y in [2019, 2020, 2021]:
    print(f'{y} AGO: cleaned Daily Instruments ${G[y].at["AGO", DI] / 1e6:,.1f}M | raw ch.89 ${raw_ch(y, [89]).get("AGO", 0) / 1e6:,.1f}M | '
          f'raw ch.89-95 (rebuilt DI) ${XB["R", "L0", y].at["AGO", DI] / 1e6:,.1f}M')
# Any C* cell that went negative is set to 0, which makes the drop-any-zero filter remove that country (reported above).
for y in YEARS:
    XB['C', 'F1', y][DI] = XB['C', 'F1', y][DI].clip(lower=0.0)

 year  C* countries  raw ch.89 world $B  cleaned DI world $B  C* DI < 0 (n) C* DI < 0 (who)  R: DI==0 under F1 but >0 under L0 (n)
 2019           160               111.5               1163.2              1             AGO                                      0
 2020           154               105.5               1135.2              1             AGO                                      0
 2021           136               120.4               1357.6              1             AGO                                      0
 2022           141               108.6               1290.1              0                                                      0
 2023           154               157.1               1854.0              0                                                      0
2019 AGO: cleaned Daily Instruments $39.1M | raw ch.89 $351.5M | raw ch.89-95 (rebuilt DI) $382.7M
2020 AGO: cleaned Daily Instruments $23.8M | raw ch.89 $950.4M | raw ch.89-95 (rebuilt DI) $966.7M
2021 AGO: cleane

In [3]:
# Four variants on each base. One shared sample per base-year: COMMON and every variant's post-filter index.
VARS = {'R0': ('L0', rca_lab), 'DD': ('L0', rca_dedup), 'F1': ('F1', rca_lab), 'DF': ('F1', rca_dedup)}
VN = list(VARS)
MM, VAR, PCI = {}, {}, {}
for b in ['R', 'C']:
    for y in YEARS:
        for v, (mp, f) in VARS.items():
            MM[b, y, v] = lab_M(XB[b, mp, y], f); z, k, q, s_ = o4(MM[b, y, v]); VAR[b, y, v] = z; PCI[b, y, v] = q
CS = {(b, y): sorted(set(COMMON[y]).intersection(*[set(VAR[b, y, v].index) for v in VN])) for b in ['R', 'C'] for y in YEARS}
print('shared sample n, R:', {y: len(CS['R', y]) for y in YEARS}); print('shared sample n, C*:', {y: len(CS['C', y]) for y in YEARS})
print('post-filter matrix n 2019, R:', {v: len(VAR['R', 2019, v]) for v in VN}, '| C*:', {v: len(VAR['C', 2019, v]) for v in VN})

shared sample n, R: {2019: 105, 2020: 105, 2021: 94, 2022: 105, 2023: 109}
shared sample n, C*: {2019: 104, 2020: 104, 2021: 93, 2022: 105, 2023: 109}
post-filter matrix n 2019, R: {'R0': 125, 'DD': 125, 'F1': 125, 'DF': 125} | C*: {'R0': 120, 'DD': 120, 'F1': 119, 'DF': 119}


## Results

### A. Do DD and F1 interact algebraically? (checks)

In [4]:
# A1. The DD factor (1 + a_c)/(1 + a_w), a = aggregate / 17-column total. F1 moves ch. 89 between two NON-aggregate columns,
# so the 17-column totals and the aggregate are unchanged: DF's RCAs should be F1's RCAs times the same factor as DD's.
rows = []
for b in ['R', 'C']:
    for y in YEARS:
        X0, X1 = eci.notebook_filter(XB[b, 'L0', y]), eci.notebook_filter(XB[b, 'F1', y]); idx = sorted(set(X0.index) & set(X1.index))
        X0, X1 = X0.loc[idx], X1.loc[idx]
        fac0 = rca_dedup(X0)[AGG] / rca_lab(X0)[AGG]; fac1 = rca_dedup(X1)[AGG] / rca_lab(X1)[AGG]
        ratio = (rca_dedup(X1) / rca_lab(X1)).sub(fac1, axis=0).abs().to_numpy()
        rows.append({'base': b, 'year': y, 'n both filters': len(idx), 'max |17-col total L0 - F1|': (X0.drop(columns=[AGG]).sum(1) - X1.drop(columns=[AGG]).sum(1)).abs().max(),
                     'max |factor L0 - factor F1|': (fac0 - fac1).abs().max(), 'max |RCA_DF/RCA_F1 - factor|': np.nanmax(ratio),
                     'world ch.89 share of 17-col total': (X0[DI] - X1[DI]).sum() / X0.drop(columns=[AGG]).to_numpy().sum()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.3g}'))

base  year  n both filters  max |17-col total L0 - F1|  max |factor L0 - factor F1|  max |RCA_DF/RCA_F1 - factor|  world ch.89 share of 17-col total
   R  2019             125                           0                            0                      6.66e-16                            0.00711
   R  2020             127                           0                            0                      6.66e-16                            0.00724
   R  2021             121                           0                            0                      5.55e-16                            0.00658
   R  2022             120                           0                            0                      6.66e-16                            0.00529
   R  2023             122                    0.000977                     4.44e-16                      6.66e-16                            0.00563
   C  2019             119                           0                            0                      6

**Takeaway (A1):** as expected, F1 leaves the 17-column totals and DD's factor unchanged (max differences 0, or 0.001 dollars on a ~$10¹³ total in 2023). DF's RCAs are exactly F1's RCAs times DD's factor (error ≤ 7e-16). Ch. 89 is only 0.53–0.72% of world 17-column exports. So any non-additivity below has to come from which cells sit near RCA = 1, not from the denominators.

In [5]:
# A2. Which M cells flip from R0 under DD, F1 and DF? Is DF's flip set the union of DD's and F1's (no interaction flips)?
def flips(b, y, v):
    A0, A1 = MM[b, y, 'R0'], MM[b, y, v]; idx = sorted(set(A0.index) & set(A1.index)); cols = sorted(set(A0.columns) & set(A1.columns))
    D = (A0.loc[idx, cols] != A1.loc[idx, cols]).stack()
    return set(D[D].index)
rows = []
for b in ['R', 'C']:
    for y in YEARS:
        sD, sF, sB = flips(b, y, 'DD'), flips(b, y, 'F1'), flips(b, y, 'DF')
        rows.append({'base': b, 'year': y, 'flips DD': len(sD), 'flips F1': len(sF), 'flips DF': len(sB),
                     'DD & F1 overlap': len(sD & sF), 'DF only (interaction)': len(sB - (sD | sF)),
                     'in DD|F1 but not DF': len((sD | sF) - sB),
                     'F1 flips in DI/TR cols': sum(1 for (_, g) in sF if g in (DI, TR)), 'DF countries w/ flip': len({i for (i, _) in sB})})
FL = pd.DataFrame(rows); print(FL.to_string(index=False))
y = 2019; sD, sF, sB = flips('R', y, 'DD'), flips('R', y, 'F1'), flips('R', y, 'DF')
print('\n2019 R: DF-only flips:', sorted(sB - (sD | sF)), '| cancelled (in DD or F1, not DF):', sorted((sD | sF) - sB))
print('2019 R: DD & F1 overlap cells:', sorted(sD & sF))

base  year  flips DD  flips F1  flips DF  DD & F1 overlap  DF only (interaction)  in DD|F1 but not DF  F1 flips in DI/TR cols  DF countries w/ flip
   R  2019        83        14        85                3                      1                   10                      14                    61
   R  2020        71        17        77                4                      1                    8                      17                    57
   R  2021        72        19        79                2                      1                   11                      19                    57
   R  2022        63         6        67                0                      1                    3                       6                    51
   R  2023        74        17        80                4                      0                    7                      17                    59
   C  2019        80        12        79                3                      2                   12           

**Takeaway (A2):** on R in 2019, DD flips 83 cells and F1 flips 14 (all 14 in *Daily Instruments*/*Transportation*). 3 cells are flipped by both (Congo, Mexico and Nigeria *Daily Instruments*). DF flips 85. **10 cells that one fix flips are back at their R0 value in DF**, and only 1 cell (Slovenia *Daily Instruments*) flips only under DF. The cancelled cells are all in the two F1 columns: Austria, Gabon and Slovenia *Transportation*; Hong Kong, Italy, Japan, Malta, Portugal, Tunisia and Vietnam *Daily Instruments*. Removing ch. 89 moves those cells' RCA in the opposite direction to DD's factor, so the two fixes partly offset each other on the same thresholds. The pattern repeats in every year (3–12 cancelled cells, 0–3 interaction cells) and on C\*.

### B. Rule A: DF against R0, F1 and DD

In [6]:
# B1. Metrics for every variant on the shared sample, both bases.
rows = []
for b in ['R', 'C']:
    for y in YEARS:
        c = CS[b, y]
        for v in VN:
            e = VAR[b, y, v][c]
            rows.append({'base': b, 'year': y, 'variant': v, 'n': len(c), 'M1': stats.spearmanr(e, ATL[y][c])[0], 'M2': stats.spearmanr(e, LNY[y][c])[0],
                         'rho w/ R0': stats.spearmanr(e, VAR[b, y, 'R0'][c])[0], 'M3 HS4 (report only)': stats.spearmanr(e, HB[y][0][c])[0]})
P = pd.DataFrame(rows)
for m in ['M1', 'M2']:
    P[m + ' diff vs R0'] = P[m] - P.groupby(['base', 'year'])[m].transform(lambda s: s.iloc[0])
print(P.round(3).to_string(index=False))
P.round(4).to_csv(FIG / 'primary_metrics.csv', index=False)

base  year variant   n    M1    M2  rho w/ R0  M3 HS4 (report only)  M1 diff vs R0  M2 diff vs R0
   R  2019      R0 105 0.689 0.453      1.000                 0.750          0.000          0.000
   R  2019      DD 105 0.716 0.447      0.931                 0.739          0.027         -0.006
   R  2019      F1 105 0.726 0.487      0.983                 0.778          0.037          0.034
   R  2019      DF 105 0.717 0.442      0.938                 0.742          0.028         -0.010
   R  2020      R0 105 0.760 0.508      1.000                 0.782          0.000          0.000
   R  2020      DD 105 0.765 0.515      0.949                 0.778          0.005          0.007
   R  2020      F1 105 0.787 0.519      0.973                 0.801          0.026          0.011
   R  2020      DF 105 0.774 0.513      0.937                 0.777          0.013          0.005
   R  2021      R0  94 0.789 0.465      1.000                 0.802          0.000          0.000
   R  2021      DD  

**Takeaway (B1):** the reproduction checks pass. On R, F1 vs R0 is +0.037 / +0.034 in 2019, matching 11-25. DD vs R0 is +0.027 / −0.006, matching 11-27. On C\* (104 countries; Angola drops out, see Setup), DD is +0.052 / +0.023, against 11-27's +0.051 / +0.023 on 105. DF lands *between* the two single fixes rather than above them. On R in 2019 it has M1 0.717 (DD 0.716, F1 0.726) and M2 0.442, which is **below R0's 0.453**.

In [7]:
# B2. Rule A verdicts for every (variant, baseline) pair the method pre-states, both bases, + 2019 paired country bootstrap (2,000 draws).
PAIRS = [('DF', 'R0'), ('DF', 'F1'), ('DF', 'DD'), ('F1', 'R0'), ('DD', 'R0')]
rng = np.random.default_rng(20261128); B = 2000
piv = {m: P.pivot_table(index=['base', 'year'], columns='variant', values=m) for m in ['M1', 'M2']}
dec = []
for b in ['R', 'C']:
    c = np.array(CS[b, 2019]); at, ly = ATL[2019][c].to_numpy(), LNY[2019][c].to_numpy()
    draws = rng.integers(0, len(c), size=(B, len(c)))
    for v, base in PAIRS:
        d1 = piv['M1'].loc[b][v] - piv['M1'].loc[b][base]; d2 = piv['M2'].loc[b][v] - piv['M2'].loc[b][base]
        both = (d1 > 0) & (d2 > 0)
        e1, e0 = VAR[b, 2019, v][c].to_numpy(), VAR[b, 2019, base][c].to_numpy()
        bb = np.array([(stats.spearmanr(e1[i], at[i])[0] - stats.spearmanr(e0[i], at[i])[0],
                        stats.spearmanr(e1[i], ly[i])[0] - stats.spearmanr(e0[i], ly[i])[0]) for i in draws])
        q1, q2 = np.percentile(bb[:, 0], [2.5, 97.5]), np.percentile(bb[:, 1], [2.5, 97.5])
        dec.append({'base': b, 'variant vs baseline': f'{v} vs {base}', 'dM1 2019': round(d1[2019], 3), 'dM1 95% CI': f'[{q1[0]:.3f}, {q1[1]:.3f}]',
                    'dM2 2019': round(d2[2019], 3), 'dM2 95% CI': f'[{q2[0]:.3f}, {q2[1]:.3f}]', 'both 2019': bool(both[2019]),
                    'both, of 2020-22': int(both.loc[[2020, 2021, 2022]].sum()), 'which': ','.join(str(t) for t in [2020, 2021, 2022] if both[t]),
                    'PASS rule A': bool(both[2019] and both.loc[[2020, 2021, 2022]].sum() >= 2),
                    'dM1 2020-22': ' / '.join(f'{d1[t]:+.3f}' for t in [2020, 2021, 2022]), 'dM2 2020-22': ' / '.join(f'{d2[t]:+.3f}' for t in [2020, 2021, 2022])})
DEC = pd.DataFrame(dec); print(DEC.to_string(index=False))
DEC.to_csv(FIG / 'rule_a_verdicts.csv', index=False)

base variant vs baseline  dM1 2019      dM1 95% CI  dM2 2019      dM2 95% CI  both 2019  both, of 2020-22          which  PASS rule A              dM1 2020-22              dM2 2020-22
   R            DF vs R0     0.028 [-0.038, 0.101]    -0.010 [-0.109, 0.080]      False                 3 2020,2021,2022        False +0.013 / +0.019 / +0.002 +0.005 / +0.032 / +0.014
   R            DF vs F1    -0.009 [-0.060, 0.039]    -0.044 [-0.135, 0.035]      False                 2      2021,2022        False -0.013 / +0.014 / +0.001 -0.005 / +0.027 / +0.008
   R            DF vs DD     0.001 [-0.015, 0.018]    -0.005 [-0.028, 0.018]      False                 1           2022        False +0.008 / -0.009 / +0.004 -0.002 / -0.029 / +0.003
   R            F1 vs R0     0.037 [-0.002, 0.093]     0.034  [0.004, 0.076]       True                 3 2020,2021,2022         True +0.026 / +0.006 / +0.001 +0.011 / +0.005 / +0.006
   R            DD vs R0     0.027 [-0.042, 0.101]    -0.006 [-0.105, 0.089]    

**Takeaway (B2):** these are the pre-stated decisions.
1. **DF vs R0:** it **fails** on the primary base R (2019 ΔM2 −0.010) and passes on C\* (+0.047 / +0.022, 3 of 3 check years).
2. **DF vs F1:** it **fails on both bases**. 2019 ΔM2 is −0.044 on R and −0.012 on C\*, so DD should not be added on top of F1.
3. **DF vs DD:** it fails on both bases (2019 ΔM2 −0.005 on R, −0.001 on C\*, with 2019 CIs inside ±0.03). Once DD is in, F1 adds almost nothing.

F1 vs R0 passes on both bases. It is the only comparison with a 2019 bootstrap CI that excludes 0 (ΔM2 [0.004, 0.076] on R, [0.005, 0.076] on C\*). DD vs R0 again passes on C\* and fails on R, as in 11-27.

### C. Is the combination additive?

In [8]:
# C. Interaction = dDF - (dDD + dF1), all vs R0. Near 0 = additive; negative = the two fixes overlap (sub-additive).
rows = []
for b in ['R', 'C']:
    for y in YEARS:
        r = {'base': b, 'year': y}
        for m in ['M1', 'M2']:
            p = piv[m].loc[b].loc[y]; dD, dF, dB = p['DD'] - p['R0'], p['F1'] - p['R0'], p['DF'] - p['R0']
            r[f'{m}: dDD'] = dD; r[f'{m}: dF1'] = dF; r[f'{m}: dDF'] = dB; r[f'{m}: interaction'] = dB - dD - dF
        rows.append(r)
ADD = pd.DataFrame(rows); print(ADD.round(3).to_string(index=False))
for b in ['R', 'C']:
    s = ADD[(ADD.base == b) & ADD.year.isin(CHK)]
    print(f'{b}: mean interaction 2019-22  M1 {s["M1: interaction"].mean():+.4f}  M2 {s["M2: interaction"].mean():+.4f} | '
          f'DF > max(DD, F1) on both metrics in {int(((s["M1: dDF"] > s[["M1: dDD", "M1: dF1"]].max(1)) & (s["M2: dDF"] > s[["M2: dDD", "M2: dF1"]].max(1))).sum())} of 4 years')

base  year  M1: dDD  M1: dF1  M1: dDF  M1: interaction  M2: dDD  M2: dF1  M2: dDF  M2: interaction
   R  2019    0.027    0.037    0.028           -0.036   -0.006    0.034   -0.010           -0.039
   R  2020    0.005    0.026    0.013           -0.018    0.007    0.011    0.005           -0.013
   R  2021    0.029    0.006    0.019           -0.015    0.061    0.005    0.032           -0.034
   R  2022   -0.003    0.001    0.002            0.003    0.011    0.006    0.014           -0.003
   R  2023    0.000    0.019    0.015           -0.005   -0.023   -0.004   -0.024            0.003
   C  2019    0.052    0.032    0.047           -0.037    0.023    0.034    0.022           -0.036
   C  2020    0.007    0.030    0.018           -0.019    0.028    0.032    0.032           -0.028
   C  2021    0.033    0.012    0.035           -0.009    0.061    0.009    0.059           -0.011
   C  2022   -0.003    0.001    0.002            0.003    0.011    0.006    0.014           -0.003
   C  2023

**Takeaway (C):** the interaction is negative in every base-year from 2019 to 2021 on both metrics (for example R 2019: −0.036 on M1, −0.039 on M2). It is close to zero in 2022–2023 (−0.005 to +0.003), where F1 flips only 6 cells in 2022. Averaged over 2019–2022, the combination loses about 0.016 of M1 and about 0.02 of M2 compared with adding the two gains. DF beats the better single fix on both metrics in only 1 of 4 years. The fixes **overlap**: DD's gain in 2021 (+0.061 M2 on R) and F1's gain in 2019 (+0.034) each mostly vanish in the combination (DF: +0.032 in 2021, −0.010 in 2019).

### D. Movers

In [9]:
# D. Largest rank moves R0 -> DF (primary base), with ATLAS rank and which fix the move comes from.
for y in CHK:
    c = CS['R', y]; r0 = rank(VAR['R', y, 'R0'], c); rD, rF, rB = (rank(VAR['R', y, v], c) for v in ['DD', 'F1', 'DF']); ra = rank(ATL[y], c)
    d = (rB - r0); big = d[d.abs() >= 15].sort_values()
    toward = ((rB - ra).abs() < (r0 - ra).abs())
    print(f'{y} | n={len(c)} | median |move| R0->DF {d.abs().median():.1f} (DD {(rD - r0).abs().median():.1f}, F1 {(rF - r0).abs().median():.1f}) | '
          f'moves >= 10 toward ATLAS: {int((toward & (d.abs() >= 10)).sum())} of {int((d.abs() >= 10).sum())}')
    print('   >=15:', ', '.join(f'{i} {int(r0[i])}->{int(rB[i])} (DD {int(rD[i])}, F1 {int(rF[i])}, ATLAS {int(ra[i])})' for i in big.index))

2019 | n=105 | median |move| R0->DF 5.0 (DD 5.0, F1 1.0) | moves >= 10 toward ATLAS: 13 of 23
   >=15: RUS 76->50 (DD 43, F1 75, ATLAS 40), GUY 79->95 (DD 96, F1 78, ATLAS 81), KEN 48->64 (DD 66, F1 46, ATLAS 83), NOR 70->87 (DD 83, F1 68, ATLAS 30), PAK 60->78 (DD 81, F1 59, ATLAS 85), NGA 18->40 (DD 37, F1 39, ATLAS 104), MDA 24->51 (DD 50, F1 27, ATLAS 72), BHR 62->91 (DD 90, F1 61, ATLAS 46), BRN 63->103 (DD 104, F1 62, ATLAS 49), KWT 63->103 (DD 104, F1 62, ATLAS 60), COG 41->83 (DD 76, F1 89, ATLAS 101)
2020 | n=105 | median |move| R0->DF 4.0 (DD 4.0, F1 2.0) | moves >= 10 toward ATLAS: 7 of 18
   >=15: ZMB 75->48 (DD 49, F1 74, ATLAS 87), MDG 91->65 (DD 68, F1 88, ATLAS 101), CYP 53->36 (DD 39, F1 45, ATLAS 50), CHL 61->76 (DD 74, F1 59, ATLAS 77), TUN 26->42 (DD 43, F1 20, ATLAS 55), COG 44->87 (DD 38, F1 79, ATLAS 103), KWT 59->104 (DD 104, F1 60, ATLAS 59), NGA 50->104 (DD 104, F1 105, ATLAS 104)
2021 | n=94 | median |move| R0->DF 4.0 (DD 3.0, F1 1.0) | moves >= 10 toward ATL

**Takeaway (D):** DF's rank moves are mostly DD's. The median |move| is 5.0 in 2019, equal to DD and against 1.0 for F1. Only 13 of the 23 moves of ≥ 10 places go toward the ATLAS rank in 2019, and 4 of 8 in 2022. Large moves *away* from ATLAS come from DD. Kuwait and Brunei both go 63 → 103 (ATLAS 60 and 49), Bahrain 62 → 91 (ATLAS 46) and Norway 70 → 87 (ATLAS 30). These are rich, resource-heavy exporters, so moving them down costs both M1 and M2. Nigeria (18 → 40, ATLAS 104) and Congo (41 → 83, ATLAS 101) move toward ATLAS under both fixes.

### E. Spring 2026 report countries

In [10]:
# E. Report countries: overall rank and position among (country + 3 peers), primary base R, 2019-2022.
rows = []
for y in CHK:
    c = CS['R', y]
    for iso in R.REPORT_COUNTRIES:
        if iso not in c: continue
        grp = [i for i in [iso] + PEERS[iso] if i in c]; r = {'year': y, 'country': iso, 'peers in sample': len(grp) - 1}
        for v in VN:
            r[v + ' rank'] = int(rank(VAR['R', y, v], c)[iso]); r[v + ' pos'] = int(VAR['R', y, v][grp].rank(ascending=False)[iso])
        r['ATLAS rank'] = int(rank(ATL[y], c)[iso]); r['ATLAS pos'] = int(ATL[y][grp].rank(ascending=False)[iso])
        rows.append(r)
RP = pd.DataFrame(rows); print(RP.to_string(index=False))
for v in ['DD', 'F1', 'DF']:
    print(f'{v}: peer position changed vs R0 in {int((RP["R0 pos"] != RP[v + " pos"]).sum())} of {len(RP)} | '
          f'position equals ATLAS position: {int((RP[v + " pos"] == RP["ATLAS pos"]).sum())} (R0: {int((RP["R0 pos"] == RP["ATLAS pos"]).sum())})')
RP.to_csv(FIG / 'report_countries.csv', index=False)

 year country  peers in sample  R0 rank  R0 pos  DD rank  DD pos  F1 rank  F1 pos  DF rank  DF pos  ATLAS rank  ATLAS pos
 2019     CAN                3       82       3       74       3       79       3       74       3          33          3
 2019     DEU                3        1       1        8       1        1       1        8       1           5          1
 2019     POL                3       33       4       28       4       23       4       20       4          24          4
 2019     SGP                3       25       4       24       4       26       4       26       4           4          3
 2019     ARE                2      104       3      102       2      103       3      102       2          42          1
 2019     GBR                3       35       4       22       3       34       4       25       3          11          2
 2019     VNM                2       31       3       54       3       32       3       38       3          56          3
 2020     CAN           

**Takeaway (E):** DF changes 7 of 28 peer positions in 2019–2022 (F1 changes 2, DD changes 4). For example, the UAE goes 3 → 2 in 2019 and 2020, the UK 4 → 3 in 2019, Singapore 4 → 3 in 2020 and 2021, and Vietnam 3 → 2 in 2021. DF and F1 each match the ATLAS peer position in 19 of 28 cases, against 17 for R0 and DD. So DF's extra changes over F1 don't buy extra agreement on the reports' peer sentences. Germany falls from 1st to 8th overall under DD and DF in 2019 but stays 1st among its peers. Vietnam's 2019 rank under DF (38) sits between R0 (31) and DD (54), because F1 cancels DD's flip of its *Daily Instruments* cell (A2).

### F. Secondary (report only): rank stability

In [11]:
# F. Year-to-year Spearman on a fixed panel (countries in every 2019-2022 shared sample), primary base R.
PAN = sorted(set.intersection(*[set(CS['R', y]) for y in CHK]))
rows = []
for v in VN:
    r = {'variant': v, 'panel n': len(PAN)}
    for a, b2 in zip(CHK[:-1], CHK[1:]):
        r[f'{a}->{b2}'] = stats.spearmanr(VAR['R', a, v][PAN], VAR['R', b2, v][PAN])[0]
    r['mean'] = np.mean([r[k] for k in r if '->' in k]); rows.append(r)
print(pd.DataFrame(rows).round(3).to_string(index=False))

variant  panel n  2019->2020  2020->2021  2021->2022  mean
     R0       86       0.941       0.938       0.960 0.946
     DD       86       0.919       0.963       0.965 0.949
     F1       86       0.928       0.955       0.964 0.949
     DF       86       0.921       0.964       0.961 0.949


**Takeaway (F, report only):** on the 86-country fixed panel, mean year-to-year Spearman is 0.946 (R0) and 0.949 for DD, F1 and DF alike. None of the variants changes stability in a meaningful way.

### Figure

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
COL = {'DD': '#9a9994', 'F1': '#e8a33d', 'DF': '#2a78d6'}
for ax, m, t in zip(axes, ['M1 diff vs R0', 'M2 diff vs R0'], ['ΔM1 vs R0 (Spearman with ATLAS ECI)', 'ΔM2 vs R0 (Spearman with log GDPpc)']):
    for v in ['DD', 'F1', 'DF']:
        for b, ls in [('R', '-'), ('C', ':')]:
            s = P[(P.base == b) & (P.variant == v)]
            ax.plot(s.year, s[m], color=COL[v], lw=2 if v == 'DF' else 1.4, marker='o', ms=4, ls=ls,
                    label=f'{v} ({"rebuilt R, primary" if b == "R" else "cleaned C*"})')
    ax.axhline(0, color='#bdbcb8', lw=0.8); ax.axvspan(2022.5, 2023.5, color='#efefec', zorder=0)
    ax.set_title(t, fontsize=10, loc='left'); ax.set_xticks(YEARS); ax.grid(axis='y', color='#ececea', lw=0.8)
    for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
axes[0].set_ylabel('Δ Spearman ρ'); axes[1].legend(frameon=False, fontsize=7, ncol=2)
fig.text(0.99, 0.01, 'shaded: 2023 (suspect, D1)', ha='right', fontsize=7.5, color='#52514e')
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / 'delta_metrics.png', dpi=100); plt.close(fig)
print('saved figures/delta_metrics.png')

saved figures/delta_metrics.png


![delta metrics](figures/delta_metrics.png)

**Takeaway (figure):** F1 (orange) is above zero on both metrics in every year 2019–2022 on both bases. On the primary base R (solid), DF (blue) sits below F1 on both metrics in 2019 and 2020 and dips below zero on M2 in 2019. DD (grey) is the most base-dependent of the three.

## Caveats

- **Small differences.** Every DF comparison has a 2019 bootstrap CI that includes 0. The verdicts rest on the pre-stated rule, not on significance. The consistent signals are the negative interaction (every 2019–2021 base-year) and DF < F1 on 2019 M2 on both bases.
- **C\* is a hybrid.** It is the cleaned CSV with *raw* ch. 89 values moved between two columns. For Angola in 2019–2021 the raw ch. 89 value exceeds the whole cleaned *Daily Instruments* cell, so that cell was clipped to 0 and Angola left the C\* sample (104 / 104 / 93 countries). See the data section.
- **ATLAS is a benchmark, not ground truth**, and M2 penalises moving rich resource exporters down (11-26 G). DD's moves away from ATLAS (Kuwait, Brunei, Bahrain, Norway) hit both metrics at once.
- **Binary RCA drives the interaction.** All of the non-additivity runs through cells near RCA = 1 (A1/A2). A continuous RCA would probably make the two fixes closer to additive (11-27 proposal #3, still open).
- Nominal values. 2023 is suspect (D1) and excluded from every rule. The peer dict is the one used by earlier runs (from the Spring 2026 PDFs) and was not re-read tonight.

## Helpful to know

- **F1 and DD commute exactly at the RCA level.** Moving dollars between two non-aggregate columns leaves every country's DD factor (1 + a_c)/(1 + a_w) unchanged. So RCA_DF = RCA_F1 × factor_c. The two can be applied in either order and give the same matrix. This holds for *any* remap that keeps the aggregate's chapters fixed.
- **Fixes that act on the same columns can cancel.** F1 shifts RCA in *Daily Instruments* and *Transportation*, and DD's rescaling flips many cells in those same columns (11-27: *Daily Instruments* was the 2nd most-flipped column). Future combined variants should check flip-set overlap (cell A2 is a reusable template) before assuming the gains add up.
- **DF's rank changes are mostly DD's.** The median move is 5 places for DF and DD, 1 for F1. If the lab adopts only F1, the printed ranks change far less (2 of 28 peer positions).
- **Reproduction anchors:** F1 vs R0 on R in 2019 = +0.037 / +0.034, and DD vs R0 = +0.027 / −0.006, on the same 105-country sample as 11-24 to 11-27.

## New research avenues (also in `proposals/2026-11-28.md`)

1. **Report-impact memo for F1 (instead of DD).** F1 is now the only fix that passes on both bases and survives the combination test. For the 7 Spring 2026 countries, recompute the 2019–2023 grouped-ECI paths, direction words and peer sentences under F1, and list exactly which printed numbers would change. This is 11-27 #4 retargeted, and it's what Will needs to decide on adopting F1 in `datacleaning2025.ipynb` for Fall 2026. One night.
2. **Which countries make DD cost income fit on top of F1?** Leave-one-country-out on the 2019 DF-vs-F1 ΔM2 (−0.044 on R). Check whether the loss is Kuwait, Brunei, Bahrain and Norway (the D movers) or spread out. If it is the four resource exporters, DD's cost is the known M2/resource tension (11-26 G), not a defect. One night.
3. **Continuous RCA for the 2×2** (extends 11-27 #3). Under log(1 + RCA), DD is a smooth per-country shift and F1 a smooth two-column shift, so the threshold cancellations in A2 should disappear. Does DF become additive, and does it then beat F1? One night.
4. **A rule for implementation changes that requires passing on both bases.** 11-27 and tonight both had verdicts that flipped between the cleaned and rebuilt bases. Proposal for Will: pre-register "rule A on both C and R" for any change to `compute_mcp` or the mapping, and re-score the past variants (F1, DD, F4, RS) under it. No new computation beyond reading earlier outputs. One night.

## Data issues / data needed

- **New: Angola's ch. 89 is missing from the cleaned *Daily Instruments* cell in 2019–2021.** Cleaned *Daily Instruments* is $39.1M / $23.8M / $25.1M, but raw ch. 89 alone is $351.5M / $950.4M / $1,166.1M, and the rebuilt ch. 89–95 total is $382.7M / $966.7M / $1,210.2M (Setup cell). The cleaned value is close to the raw total *minus* ch. 89, so the cleaning step seems to have dropped Angola's ships/platforms lines in those years. 2022–2023 agree. Details: `data-requests/2026-11-28-angola-ch89-cleaned.md`. It doesn't change tonight's primary-base verdicts (R is rebuilt from raw), but it is another cleaned-vs-raw gap of the kind that makes DD's verdict base-dependent.
- Existing issues still apply: D1 (2023 inflation) and the 999999 reporters (`data-requests/2026-11-25-999999-reporters.md`, in the open 11-25 PR).

In [13]:
print(f'Run time: {time.time() - T0:.1f} s')

Run time: 25.8 s
